# ITW Mask Generator — CIFAR-10

Trains a spatial mask generator using a frozen DiT+D3PM prior.
Masked pixels use absorbing state 0; observed pixels are remapped to bins 1..N-1.

In [ ]:
from itw import (
    CIFAR10Config,
    build_dataloader,
    build_mask_model,
    build_pixel_survival_table,
    evaluate_loader,
    load_d3pm,
    plot_mask_grid,
    save_eval_report,
    train_mask_generator,
)
from itw.masks import apply_masked_observation, gumbel_mask
from itw.train import discretize, forward_mask_logits
import torch

In [ ]:
cfg = CIFAR10Config(
    device="cuda",
    mask_arch="spatial",
    n_epochs=200,
    save_every=10,
)
d3pm = load_d3pm(cfg)
model = build_mask_model(cfg)
dataloader = build_dataloader(cfg)

In [ ]:
survival_table = build_pixel_survival_table(d3pm, dataloader, device=cfg.device)

In [ ]:
model, survival_table = train_mask_generator(
    cfg,
    model=model,
    d3pm=d3pm,
    dataloader=dataloader,
    survival_table=survival_table,
)

## Evaluation

In [ ]:
metrics = evaluate_loader(
    d3pm, model, cfg, dataloader, survival_table,
    max_batches=20, fixed_sparsity=0.2,
)
print(metrics)
save_eval_report(metrics, f"{cfg.save_dir}/eval.json")

In [ ]:
x, cond = next(iter(dataloader))
sparsity = torch.full((x.shape[0],), 0.2)
x_disc = discretize(x, cfg.num_classes)
mask_logits = forward_mask_logits(model, cfg, x_disc, cond, sparsity)
mask = gumbel_mask(mask_logits, temperature=0.5, hard=True)
y = apply_masked_observation(
    x_disc, mask, cfg.num_classes, multichannel=cfg.multichannel
)
plot_mask_grid(x_disc, mask, y, cfg.num_classes, title="CIFAR-10 ITW masks")

In [ ]:
# Load checkpoint for inspection
# model.load_state_dict(torch.load(f"{cfg.save_dir}/mask_gen_cifar10_19e.pth"))